# Quick Test: does my model work?Run top to bottom. It converts your .mov files, slices them, and scores both models.## Before you start, set up Drive like this```MyDrive/theraphono/  ovs_mov/    wet_chewing/      Vid17_O.mov    gulping/          Vid18_O.mov    fork_scraping/    Vid09_O.mov    pen_clicking/     Vid20_O.mov    sniffling/        Vid12_O.mov  Vid08_O.mov  tm_model/    soundmodel.tflite    labels.txt```Which SSV video goes in which folder:| File | Sound | Folder ||---|---|---|| Vid17_O | Wet chewing | `wet_chewing` || Vid18_O | Gulping | `gulping` || Vid09_O | Fork scraping plate | `fork_scraping` || Vid20_O | Pen clicking | `pen_clicking` || Vid12_O | Sniffling | `sniffling` || Vid08_O | Sniffling | `sniffling` |Use the **_O** files (Original Video Source). Those have the real trigger sound.Skip Vid01, Vid04, Vid06, Vid07, Vid10, Vid13, Vid16. Those are slurping, crunchy chewing, squeaking,swishing, tapping and drumming, which are not classes in your model.**This is only 6 files, so it is not a real accuracy number.** It tells you whether the model works atall. Add your held-out clips later for the real evaluation.

In [ ]:
!pip install -q tensorflow tensorflow_hub librosa pandas scikit-learnimport os, glob, subprocessimport numpy as np, pandas as pdimport tensorflow as tf, tensorflow_hub as hub, librosafrom google.colab import drivedrive.mount('/content/drive')ROOT = '/content/drive/MyDrive/theraphono'MOV_DIR   = f'{ROOT}/ovs_mov'TM_MODEL  = f'{ROOT}/tm_model/soundmodel.tflite'TM_LABELS = f'{ROOT}/tm_model/labels.txt'CLASSES = ['wet_chewing','gulping','fork_scraping','pen_clicking',           'sniffling','speech','other']print("ok")

## Step 1: .mov to 1-second WAV clipsPulls the audio out of each video, normalizes the volume, and cuts it into 1-second pieces.Same treatment as your training clips.

In [ ]:
WORK = '/content/test_clips'!rm -rf {WORK}os.makedirs(WORK, exist_ok=True)rows = []for cls in CLASSES:    src_dir = os.path.join(MOV_DIR, cls)    if not os.path.isdir(src_dir):        continue    out_dir = os.path.join(WORK, cls)    os.makedirs(out_dir, exist_ok=True)    for f in sorted(os.listdir(src_dir)):        if not f.lower().endswith(('.mov','.mp4','.m4v','.avi','.wav','.mp3')):            continue        src = os.path.join(src_dir, f)        stem = os.path.splitext(f)[0]        full = os.path.join(WORK, f'_{cls}_{stem}.wav')        # extract audio, mono, 44.1kHz, loudness normalized        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',src,'-vn',                        '-af','loudnorm=I=-23:LRA=7:TP=-2',                        '-ac','1','-ar','44100', full], check=True)        dur = float(subprocess.run(            ['ffprobe','-v','error','-show_entries','format=duration',             '-of','csv=p=0', full], capture_output=True, text=True).stdout.strip())        n = 0        s = 0.0        while s + 1.0 <= dur + 1e-6:            clip = os.path.join(out_dir, f'{stem}_c{n:03d}.wav')            subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',f'{s:.3f}',                            '-t','1.0','-i',full,'-ac','1','-ar','44100', clip], check=True)            rows.append({'path':clip,'true_label':cls,'source':stem})            n += 1            s += 1.0        os.remove(full)        print(f'{cls:<15} {f:<20} {dur:5.1f}s -> {n} clips')df = pd.DataFrame(rows)print()print(len(df), 'test clips total')print(df['true_label'].value_counts())

## Step 2: YAMNetNo training. It already knows 521 sound categories. Let's see what it says.

In [ ]:
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')yc = pd.read_csv(yamnet.class_map_path().numpy().decode())['display_name'].tolist()# What does YAMNet think one chewing clip is?p = df[df.true_label=='wet_chewing'].iloc[0]['path']w, _ = librosa.load(p, sr=16000, mono=True)sc, _, _ = yamnet(w.astype(np.float32))m = np.mean(sc.numpy(), axis=0)print('TOP LABELS FOR ONE CHEWING CLIP:')for i in np.argsort(m)[::-1][:10]:    print(f'  {m[i]:.4f}  {yc[i]}')

**Screenshot that.** If Chewing or Biting are near the top, YAMNet recognizes yourtrigger with zero training. That is the single most useful thing you can show your mentor.

In [ ]:
LABEL_MAP = {    'wet_chewing':   ['Chewing, mastication','Biting'],    'gulping':       ['Slurp','Gargling'],    'fork_scraping': ['Scrape','Squeak'],    'pen_clicking':  ['Click','Tick','Typing','Computer keyboard'],    'sniffling':     ['Sniff','Snort','Breathing'],    'speech':        ['Speech','Conversation'],}bad = [n for v in LABEL_MAP.values() for n in v if n not in yc]print('not real YAMNet labels, remove these:', bad) if bad else print('map ok')idx = {n:i for i,n in enumerate(yc)}THRESHOLD = 0.05def yamnet_predict(path):    w, _ = librosa.load(path, sr=16000, mono=True)    sc, _, _ = yamnet(w.astype(np.float32))    m = np.mean(sc.numpy(), axis=0)    best, bs = 'other', 0.0    for cls, names in LABEL_MAP.items():        s = max(m[idx[n]] for n in names if n in idx)        if s > bs: best, bs = cls, s    return ('other' if bs < THRESHOLD else best), float(bs)df['yamnet_pred'] = [yamnet_predict(p)[0] for p in df['path']]print('done')

## Step 3: Teachable Machine**Read the shape this prints.** It decides whether the rest works.

In [ ]:
interp = tf.lite.Interpreter(model_path=TM_MODEL)interp.allocate_tensors()inp, out = interp.get_input_details(), interp.get_output_details()print('INPUT :', inp[0]['shape'], inp[0]['dtype'])print('OUTPUT:', out[0]['shape'])tm_labels = [l.strip().split(' ',1)[-1] for l in open(TM_LABELS) if l.strip()]print('labels:', tm_labels)

- **`[1, 44032]` or similar 2-D** → raw audio. Keep going.- **`[1, 43, 232, 1]` or any 3-D/4-D** → it wants a spectrogram. Stop here, you cannot feed it  without reproducing Teachable Machine's exact settings. Screenshot the shape and tell your mentor.  You still have the YAMNet result, which is the more important half.

In [ ]:
TM_LEN = int(inp[0]['shape'][-1])def tm_predict(path):    w, _ = librosa.load(path, sr=44100, mono=True)    w = w.astype(np.float32)    if len(w) < TM_LEN: w = np.pad(w, (0, TM_LEN-len(w)))    interp.set_tensor(inp[0]['index'], w[:TM_LEN].reshape(1,-1))    interp.invoke()    pr = interp.get_tensor(out[0]['index'])[0]    return tm_labels[int(np.argmax(pr))], float(np.max(pr))# Rename Teachable Machine's labels to match CLASSES if they differ.TM_TO_CANON = {'Background Noise':'other'}df['tm_pred'] = [TM_TO_CANON.get(tm_predict(p)[0], tm_predict(p)[0]) for p in df['path']]print('done')

## Step 4: The numbers

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_scorepresent = [c for c in CLASSES if c in df['true_label'].unique()]for name, col in [('YAMNET (untrained)','yamnet_pred'), ('TEACHABLE MACHINE','tm_pred')]:    print('='*60); print(name); print('='*60)    print('accuracy:', round(accuracy_score(df.true_label, df[col]),3),          '  macro F1:', round(f1_score(df.true_label, df[col], average='macro', zero_division=0),3))    print()    print(classification_report(df.true_label, df[col], labels=present, zero_division=0))    print('rows=TRUE  cols=PREDICTED')    print(pd.DataFrame(confusion_matrix(df.true_label, df[col], labels=present),                       index=present, columns=present))    print()

In [ ]:
# Per-clip side by side. Useful for spotting patterns.df[['source','true_label','yamnet_pred','tm_pred']]

In [ ]:
df.to_csv(f'{ROOT}/quick_test_results.csv', index=False)print('saved to Drive')

## What this does and does not tell you**Does:** whether either model recognizes your triggers at all, and which sounds get confused.**Does not:** a trustworthy accuracy number. Six source files is far too few, and there is no `speech`or `other` in here, so you cannot measure false alarms. That needs your held-out clips.## For the meeting- YAMNet's top labels on the chewing clip- Both confusion matrices- The Teachable Machine input shape- That this is 6 files and you know it is not enough yet